# Generate Training Data with InstructLab

This notebook uses InstructLab to generate synthetic training data from taxonomy.

## 1. Install Dependencies

In [ ]:
!pip install instructlab

## 2. Initialize InstructLab

In [ ]:
!ilab config init --non-interactive

## 3. Verify Taxonomy Files

In [ ]:
import os
from pathlib import Path

taxonomy_files = [
    '../taxonomy/security/qna.yaml',
    '../taxonomy/recruiter/qna.yaml'
]

print("Checking taxonomy files:\n")
for file in taxonomy_files:
    exists = Path(file).exists()
    status = "✅" if exists else "❌"
    print(f"{status} {file}")

if all(Path(f).exists() for f in taxonomy_files):
    print("\n✅ All taxonomy files found!")
else:
    print("\n❌ Some taxonomy files are missing. Please create them first.")

## 4. Generate Synthetic Training Data

In [ ]:
%%time

!ilab data generate \
    --taxonomy-path ../taxonomy \
    --num-instructions 100 \
    --output-dir ../datasets/generated \
    --model-name ibm-granite/granite-3b-code-instruct

## 5. Inspect Generated Data

In [ ]:
import json
import glob

generated_files = glob.glob('../datasets/generated/*.jsonl')

print(f"Found {len(generated_files)} generated files:\n")
for file in generated_files:
    print(f"  📄 {file}")

if generated_files:
    print("\nLoading first file for inspection...")
    
    with open(generated_files[0], 'r') as f:
        examples = [json.loads(line) for line in f]
    
    print(f"Total examples: {len(examples)}")
    print("\nFirst example:")
    print(json.dumps(examples[0], indent=2)[:500] + "...")

## 6. Combine Data for Multi-task Training

In [ ]:
import json
import random
from pathlib import Path

print("Combining training data from all taxonomy categories...\n")

all_examples = []

security_files = glob.glob('../datasets/generated/*security*.jsonl')
for file in security_files:
    with open(file, 'r') as f:
        examples = [json.loads(line) for line in f]
        all_examples.extend(examples)
        print(f"Loaded {len(examples)} security examples from {Path(file).name}")

recruiter_files = glob.glob('../datasets/generated/*recruiter*.jsonl')
for file in recruiter_files:
    with open(file, 'r') as f:
        examples = [json.loads(line) for line in f]
        all_examples.extend(examples)
        print(f"Loaded {len(examples)} recruiter examples from {Path(file).name}")

random.shuffle(all_examples)

output_file = '../datasets/generated/combined_training_data.jsonl'
with open(output_file, 'w') as f:
    for example in all_examples:
        f.write(json.dumps(example) + '\n')

print(f"\n✅ Combined dataset created: {output_file}")
print(f"Total examples: {len(all_examples)}")

## 7. Data Quality Check

In [ ]:
import json

with open('../datasets/generated/combined_training_data.jsonl', 'r') as f:
    examples = [json.loads(line) for line in f]

print("Data Quality Report")
print("=" * 60)
print(f"Total examples: {len(examples)}")

valid_count = 0
for ex in examples:
    if 'messages' in ex and len(ex['messages']) >= 2:
        valid_count += 1

print(f"Valid format: {valid_count}/{len(examples)} ({valid_count/len(examples)*100:.1f}%)")

prompt_lengths = []
response_lengths = []

for ex in examples:
    if 'messages' in ex:
        for msg in ex['messages']:
            if msg['role'] == 'user':
                prompt_lengths.append(len(msg['content']))
            elif msg['role'] == 'assistant':
                response_lengths.append(len(msg['content']))

if prompt_lengths:
    print(f"\nAverage prompt length: {sum(prompt_lengths)/len(prompt_lengths):.0f} chars")
if response_lengths:
    print(f"Average response length: {sum(response_lengths)/len(response_lengths):.0f} chars")

print("\n✅ Data ready for fine-tuning!")

## Summary

✅ Training data generated successfully!

Next steps:
1. Review the generated examples in `../datasets/generated/`
2. Proceed to `2_finetune_granite.ipynb` for model training
3. Upload trained model to S3 for serving